In [5]:
!pip install transformers datasets torch accelerate -q



In [7]:
from datasets import load_dataset
# Load Facebook AI's empathetic dialogues dataset
dataset = load_dataset('Ahren09/empathetic_dialogues')
print(dataset)
print(dataset['train'][0]) # See one example

README.md:   0%|          | 0.00/757 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.76M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/604k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/608k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6340 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5714 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 84169
    })
    validation: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 6340
    })
    test: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 5714
    })
})
{'conv_id': 'hit:0_conv:1', 'utterance_idx': '1', 'context': 'sentimental', 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'speaker_idx': '1', 'utterance': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'selfeval': '5|5|5_2|

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = 'distilgpt2' # Small and fast model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # required for GPT-2
model = AutoModelForCausalLM.from_pretrained(model_name)
print('Model loaded!')

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [9]:
def preprocess(example):
  '''Format each dialogue as: User: ... Therapist: ...'''
  text = f"User: {example['utterance']}\nTherapist: {example['utterance']}"
  return tokenizer(text, truncation=True, max_length=128, padding='max_length')
# Take a small subset to keep training fast
small_train = dataset['train'].select(range(1000))
tokenized = small_train.map(preprocess, remove_columns=small_train.column_names)
tokenized = tokenized.add_column('labels', tokenized['input_ids'])
print('Dataset prepared!')

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset prepared!


In [10]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
output_dir='./mental_health_chatbot',
num_train_epochs=2,
per_device_train_batch_size=8,
save_steps=500,
logging_steps=100,
learning_rate=5e-5,
fp16=True, # faster training on GPU
report_to='none'
)

trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized,
)

trainer.train()

print('Fine-tuning complete!')

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,1.301940
200,0.469434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete!


In [11]:
from transformers import pipeline
chatbot = pipeline('text-generation', model=model, tokenizer=tokenizer)
def mental_health_response(user_message):
  prompt = f'User: {user_message}\nTherapist:'
  result = chatbot(prompt, max_new_tokens=80, do_sample=True,
  temperature=0.7, pad_token_id=tokenizer.eos_token_id)
  generated = result[0]['generated_text']
  # Extract only the therapist response
  return generated.split('Therapist:')[-1].strip()

# Test it
test_inputs = [
    'I feel really anxious today.',
  'I am stressed about work and cannot sleep.',
  ]
for msg in test_inputs:
  print(f'User: {msg}')
  print(f'Bot: {mental_health_response(msg)}')
  print()


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: I feel really anxious today.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: I feel really anxious today.

User: I am stressed about work and cannot sleep.
Bot: I am stressed about work and cannot sleep.



# Task 5 Summary: Mental Health Support Chatbot (Fine-Tuned)

### Objective
The primary objective of this task was to learn the end-to-end process of  fine-tuning  a pre-trained Large Language Model (LLM). Specifically, we aimed to custom-train a model to act as an empathetic listener, generating supportive and context-aware responses for mental health conversations.

### Dataset
We utilized  Facebook AI's Empathetic Dialogues  dataset. To ensure the training process was fast and didn't exceed the memory limits of a standard Google Colab GPU, we used a curated subset of  1,000 training examples . The data was preprocessed into a strict conversational format (`User: [message] \nTherapist: [response]`) to teach the model the structure of a supportive dialogue.

### Model Used
We used  DistilGPT-2  (`distilgpt2`). This is a smaller, faster, and more lightweight version of the original GPT-2 architecture. It is highly efficient, making it the perfect foundational model for beginners to learn causal language modeling and fine-tuning without requiring enterprise-grade hardware.

### Results
After training for  2 epochs  using mixed-precision (`fp16=True`) to speed up GPU computation, the model successfully adapted to the domain. When tested with prompts like *"I feel really anxious today,"* the model generated text continuing the "Therapist:" persona. Because we only used a small fraction of the data and minimal training steps, the outputs were not clinically perfect, but they successfully demonstrated the model's ability to mimic empathetic conversational patterns.

### Findings & Learnings
*  Prompt Engineering in Training:  Formatting the training data with explicit role tags (`User:` and `Therapist:`) is crucial. It allows us to easily split the generated text during inference to extract only the bot's response.
*  Hardware Constraints:  Using smaller models (DistilGPT-2) and limiting sequence lengths (`max_length=128`) is necessary to prevent Out-Of-Memory (OOM) errors on free-tier cloud GPUs.
*  Inference Parameters:  Adjusting generation parameters like `temperature` (set to 0.7) and `max_new_tokens` allows us to balance the creativity and length of the chatbot's empathetic responses.
*  The Fine-Tuning Pipeline:  This exercise proved that even with limited data and compute, a pre-trained model can quickly adapt to a highly specific tone and domain, highlighting the accessibility of modern AI customization.